# Quickstart — Your First Fine-Tune in 4 Hours

The Colab equivalent of Recipe 1 in [`docs/COOKBOOK.md`](../docs/COOKBOOK.md). Takes you from `pip install` to a fine-tuned LoRA adapter ready to serve, in one notebook.

**Runtime:** ~3 hours on a Colab A100. **Cost:** ~$2. **GPU:** A100 recommended; T4 works but takes longer.

**What you'll have at the end:**
- A customer-support fine-tune of Qwen 3.5 0.8B + LoRA r=16
- Trained on the bundled 24-scenario corpus (replace with your data)
- A reproducibility-locked result with seed=42 + commit hash recorded

**Open in Colab:** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stateset/stateset-agents/blob/master/notebooks/quickstart_first_finetune.ipynb)

## 1. Install

In [ ]:
import os
import subprocess

PINNED_COMMIT = '14c0e65'
if not os.path.exists('/content/stateset-agents'):
    subprocess.check_call([
        'git', 'clone', '--quiet',
        'https://github.com/stateset/stateset-agents',
        '/content/stateset-agents'
    ])
subprocess.check_call(['git', '-C', '/content/stateset-agents', 'checkout', '--quiet', PINNED_COMMIT])
%cd /content/stateset-agents
%pip install --quiet -e '.[training,api]'
print('Install complete')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  device: {torch.cuda.get_device_name(0)}')

## 2. Scaffold a customer-support project

This mirrors `stateset-agents starter customer-support ./client-acme --client-name "Acme Corp"` from the cookbook.

In [ ]:
result = subprocess.run([
    'stateset-agents', 'starter', 'customer-support', '/content/client-acme',
    '--client-name', 'Acme Corp',
], check=True, capture_output=True, text=True)
print(result.stdout)

# What the scaffold landed:
import os
for f in sorted(os.listdir('/content/client-acme')):
    print(f'  {f}')

## 3. Inspect (or edit) the bundled scenarios

In production, this is where you'd replace `scenarios.jsonl` with your client's real customer queries. For this demo we use the bundled 8 scenarios across 4 intents.

In [ ]:
import json
scenarios_path = '/content/client-acme/scenarios.jsonl'
scenarios = [json.loads(line) for line in open(scenarios_path) if line.strip()]
print(f'{len(scenarios)} scenarios across intents: {set(s["intent"] for s in scenarios)}')
print()
for s in scenarios[:3]:
    print(f'  [{s["intent"]}] {s["user_query"]}')
    print(f'    must_acknowledge: {s["must_acknowledge"]}')
    print(f'    must_avoid:       {s["must_avoid"]}')
    print()

## 4. Train

This runs the scaffolded `train.py` — multi-turn GSPO with LoRA r=16 on Qwen 3.5 0.8B. Takes ~3 hours on Colab A100.

In [ ]:
%cd /content/client-acme
import time
t0 = time.time()
# Live-streamed output via shell — Colab handles this well.
!python train.py 2>&1 | tee /content/train.log | tail -40
print(f'\nWall-clock: {(time.time() - t0) / 60:.1f} minutes')

## 5. Sanity-check at the REPL

Chat with the fine-tune in-process. With `--grade customer_support`, you'll see the composite-reward score after every assistant turn — same reward used during training.

In [ ]:
import asyncio
from stateset_agents.core.agent import MultiTurnAgent
from stateset_agents.core.agent_config import AgentConfig
from stateset_agents.core.trajectory import ConversationTurn
from stateset_agents.data.customer_support_bench import SupportRewardComposite

agent = MultiTurnAgent(AgentConfig(
    model_name='Qwen/Qwen3.5-0.8B',
    peft_path='/content/client-acme/outputs/acme_corp_v1',
    max_new_tokens=320,
    temperature=0.0,
    do_sample=False,
    torch_dtype='bfloat16',
))
await agent.initialize()

reward = SupportRewardComposite()

test_prompts = [
    ('I want a refund for my recent purchase',
     {'intent': 'refund', 'must_acknowledge': ['refund', 'order'], 'must_avoid': ['impossible']}),
    ('The mobile app keeps crashing',
     {'intent': 'technical', 'must_acknowledge': ['app', 'crash'], 'must_avoid': ['your fault']}),
    ('Why is my bill different this month?',
     {'intent': 'billing', 'must_acknowledge': ['bill'], 'must_avoid': ['impossible']}),
]

for prompt, context in test_prompts:
    response = await agent.generate_response(
        f'You are a helpful customer support agent.\n\nUser: {prompt}\n\nAgent:'
    )
    score = (await reward.compute_reward(
        [ConversationTurn(role='assistant', content=response)], context=context
    )).score
    marker = '✅' if score >= 0.7 else ('⚠️ ' if score >= 0.3 else '❌')
    print(f'\n{marker} reward={score:.3f}')
    print(f'User: {prompt}')
    print(f'Agent: {response[:200]}')

## 6. Save provenance for hand-off

Mirrors Cookbook Recipe 6 — package everything a colleague needs to reproduce your result.

In [ ]:
import json
from pathlib import Path

result = subprocess.run(['stateset-agents', 'version', '--json'],
                        capture_output=True, text=True, check=True)
version_info = json.loads(result.stdout)
Path('/content/client-acme/version.json').write_text(json.dumps(version_info, indent=2))
print('Saved version provenance:')
print(json.dumps(version_info, indent=2))

# The artifacts a colleague needs:
print('\nArtifacts to share:')
for f in ['config.yaml', 'scenarios.jsonl', 'reward.py', 'train.py', 'version.json']:
    p = Path(f'/content/client-acme/{f}')
    if p.exists():
        print(f'  {p}  ({p.stat().st_size} bytes)')

print('\nThe trained adapter:')
adapter = Path('/content/client-acme/outputs/acme_corp_v1')
if adapter.exists():
    for p in adapter.iterdir():
        print(f'  {p.name}')

## 7. Next steps

You now have:

- A trained LoRA adapter at `/content/client-acme/outputs/acme_corp_v1/`
- A reproducibility-locked result (seed=42, commit=14c0e65)
- A provenance manifest in `version.json`

**To iterate:** capture real conversations → grade them → curate high-scoring examples → SFT on those → next adapter. See `notebooks/grade_and_curate_demo.ipynb` and `notebooks/sft_from_curated_demo.ipynb`.

**To serve in production:** package the adapter into a Docker image with the bundled Helm chart, or hit it locally via `stateset-agents serve --checkpoint outputs/acme_corp_v1 --base-model Qwen/Qwen3.5-0.8B`.

**To look up other recipes:** `stateset-agents recipe list`.